# Setup

## Imports

In [7]:
import sys, os
import pandas as pd
import AAnF_Benchmark_Tool
from AAnF_Benchmark_Tool import AAnF_Benchmark
from AAnF_Benchmark_Tool import *

# Suppress specific warnings
import warnings
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


pd.options.display.float_format = format_float
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows"   , 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x) # Decimal in views

## Dados

In [26]:
df_raw = pd.read_csv( "20250613_AF_brazil_project_santander_pairing_phase_5_santander_analysis_cube_pj_peer.csv", sep = ",")

In [27]:
df_raw.columns

Index(['month_year', 'issuer_name', 'issuer_id', 'super_industry_name_grouped',
       'periodo_txn', 'auth_response_code', 'flag_domestic',
       'card_product_type_group', 'card_product_type', 'transaction_type',
       'pan_entry_mode', 'cvc2_presency', 'de48s87_cvv2_resp_cd',
       'flg_recurring', 'fraud_type', 'token_ind', 'range_score_di',
       'zero_dolar_flag', 'wallet_flag', 'dti_score', 'click_to_pay',
       'total_amount', 'appr_txns', 'appr_amount', 'declined_txns',
       'declined_amount', 'qt_fraud', 'amount_fraud', 'total_txns',
       'super_industry_name_grouped_new', 'ticket_range'],
      dtype='object')

In [9]:
df_raw.issuer_name.value_counts()

issuer_name
BANCO SANTANDER (BRASIL) S.A.    1161598
ITAU UNIBANCO S.A.                797290
BANCO INTER S.A.                  659336
BANCO C6 SA                       465983
BANCO BRADESCO S.A.               139360
BANCO BTG PACTUAL SA               21744
Name: count, dtype: int64

In [10]:
df_raw.loc[df_raw.issuer_name == 'BANCO SANTANDER (BRASIL) S.A.'].to_csv('20250613_AF_brazil_project_santander_pairing_phase_5_santander_analysis_cube_pj_santander.csv')

In [11]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3245311 entries, 0 to 3245310
Data columns (total 31 columns):
 #   Column                           Dtype  
---  ------                           -----  
 0   month_year                       object 
 1   issuer_name                      object 
 2   issuer_id                        int64  
 3   super_industry_name_grouped      object 
 4   periodo_txn                      object 
 5   auth_response_code               int64  
 6   flag_domestic                    object 
 7   card_product_type_group          object 
 8   card_product_type                object 
 9   transaction_type                 object 
 10  pan_entry_mode                   object 
 11  cvc2_presency                    int64  
 12  de48s87_cvv2_resp_cd             object 
 13  flg_recurring                    int64  
 14  fraud_type                       object 
 15  token_ind                        object 
 16  range_score_di                   object 
 17  zero_dol

In [12]:
df_raw['super_industry_name_grouped'] = df_raw['super_industry_name_grouped_new']
df_raw = df_raw.drop(columns=['super_industry_name_grouped_new'])

In [13]:
df_raw["de48s87_cvv2_resp_cd"] = df_raw["de48s87_cvv2_resp_cd"].fillna("NULL")
df_raw["de48s87_cvv2_resp_cd"].value_counts(dropna=False)

de48s87_cvv2_resp_cd
NULL    1849034
M       1070322
P        226701
N         99250
Y             4
Name: count, dtype: int64

In [14]:
df_raw = df_raw.loc[~df_raw['pan_entry_mode'].isin(['UNKNOWN POI','MAGNETIC STRIPE'])]

In [15]:
(
    df_raw
    .groupby(['month_year'])
    .total_txns
    .sum()
)

month_year
2025-01    7770582
2025-02    7776738
2025-03    8521801
2025-04    8621411
Name: total_txns, dtype: int64

In [16]:
df_raw = df_raw.rename(columns={
    "total_amount": "txn_amt",
    "total_txns": "txn_cnt",
    "appr_amount": "app_amt",
    "appr_txns": "app_cnt",
    "declined_amount": "dcl_amt",
    "declined_txns": "dcl_cnt",
    "qt_fraud": "fraud_cnt",
    "amount_fraud": "fraud_amt"
})

df_raw.describe()

,issuer_id,auth_response_code,cvc2_presency,flg_recurring,click_to_pay,txn_amt,app_cnt,app_amt,dcl_cnt,dcl_amt,fraud_cnt,fraud_amt,txn_cnt
count,3245026.00,3245026.00,3245026.00,3245026.00,891.00,3245026.00,3245026.00,3245026.00,3245026.00,3245026.00,3245026.00,3245026.00,3245026.00
mean,11912.84,25.92,1.59,0.23,2.00,8859.56,6.65,3653.95,3.42,5205.61,0.02,13.42,10.07
std,6239.16,28.18,0.49,0.42,0.00,4173857.04,95.30,71049.76,36.89,4173256.83,0.37,459.96,101.97
min,0.00,0.00,1.00,0.00,2.00,0.01,0.00,0.00,0.00,0.00,0.00,0.00,1.00
25%,6282.00,0.00,1.00,0.00,2.00,97.23,0.00,0.00,0.00,0.00,0.00,0.00,1.00
50%,8328.00,4.00,2.00,0.00,2.00,391.67,0.00,0.00,1.00,14.90,0.00,0.00,1.00
75%,19622.00,57.00,2.00,0.00,2.00,1674.13,1.00,423.73,1.00,367.41,0.00,0.00,4.00
max,31474.00,96.00,2.00,1.00,2.00,5315581209.87,19439.00,22668199.78,6914.00,5315581209.87,76.00,278107.70,19439.00


## Funções

In [17]:
import pandas as pd
import re
import time

def generate_columns_to_sum(columns_pivot):
    return [f"{col}_cnt" for col in columns_pivot]

def pivot_and_aggregate(df_raw, breaks, rows, columns_pivot, issuer_column):
    break_cols = [issuer_column] + breaks
    columns_to_sum = [f"{col}_cnt" for col in columns_pivot]

    # Aggregate
    df_pivot = df_raw.groupby(break_cols).agg({col: "sum" for col in columns_to_sum}).reset_index()
    # Pivot
    df_pivot_adj = pd.pivot_table(
        df_pivot,
        index=rows,
        columns=breaks,
        values=columns_to_sum,
        aggfunc="sum"
    ).reset_index().fillna(0)

    # Flatten MultiIndex columns
    df_pivot_adj.columns = df_pivot_adj.columns \
        .to_flat_index() \
        .map( lambda x: x[1:] + ( x[0], ) ) \
        .map( "_".join) \
        .str.lstrip( "_")

    # Compute row totals
    # row_totals = df_pivot_adj.drop(columns=rows).groupby(lambda x: x.split("_")[0], axis=1).sum()
    row_totals = df_pivot_adj.drop(columns=rows).groupby(lambda x: "_".join(x.split('_')[len(breaks):]), axis=1).sum()
    row_totals.columns = [f"General_{col}" for col in row_totals.columns]

    # Merge totals back
    df_pivot_adj = pd.concat([df_pivot_adj[rows], df_pivot_adj.drop(columns=rows), row_totals], axis=1)
    return df_pivot_adj

def balance_peer(df_raw, rows, breaks, columns_pivot, dict2convert, issuer_column, issuer_name,
                 rules, num_combinations, num_participants, max_percentage, selected_comb):
    print(f'\n==================================\n\nBrakes to process:', breaks)
    df_raw = df_raw.copy()
    df_pivot_adj = pivot_and_aggregate(df_raw, breaks, rows, columns_pivot, issuer_column)
    benchmark_tool = prepare_benchmark(df_pivot_adj, breaks, dict2convert, issuer_column, issuer_name)

    initial_df = benchmark_tool.initial_df
    initial_df.columns = initial_df.columns.str.strip()
    vars_to_eval = [col for col in initial_df.columns if col.startswith("v_")]

    df_kpis = evaluate_combinations(
        initial_df,
        vars_to_eval,
        rules,
        num_participants,
        max_percentage,
        num_combinations,
        selected_comb
    )

    return df_kpis


def prepare_benchmark(df_pivot_adj, break_cols, dict2convert, issuer_column, issuer_name):
    breaks_unfiltered = list(df_pivot_adj.columns)[1:]

    test = breaks_unfiltered[0]
    "".join(test)
    breaks = ["General"] + ["_".join(break_var.split("_")[:len(break_cols)]) for break_var in breaks_unfiltered[:len(breaks_unfiltered) - 3]]
    breaks = list(dict.fromkeys(breaks))

    benchmark_tool = AAnF_Benchmark(
        iss_name=issuer_name,
        iss_var_name=issuer_column,
        cnt_amt_flag="Cnt",
        breaks=breaks
    )

    print( benchmark_tool.display_attributes() ) 

    benchmark_tool.df_2_input(df_pivot_adj ,
                                dict2convert ,
                                agg_by = issuer_column
                                )

    benchmark_tool.df_input

    benchmark_tool.df_2_breaks(dummy_fraud = False)
    benchmark_tool.get_peer_distances( dict_compare = { "General_Cnt_tl0" : ["Perc_CNP_Cnt"] } ) 
    benchmark_tool.get_initial_df()

    print(benchmark_tool.df_input.shape , "\n" ,
      benchmark_tool.df_input_fraud.shape , "\n" ,
      benchmark_tool.df_breaks.shape , "\n" ,
      benchmark_tool.initial_df.shape , "\n" ,
      benchmark_tool.breaks
     )

    print([ ( i, c )  for i,c in enumerate( benchmark_tool.initial_df.columns ) if c.startswith( "v_") ])
    print('initial_df')

    return benchmark_tool

def evaluate_combinations(initial_df, vars_to_eval, rules, num_participants, max_percentage, num_combinations, selected_comb):
    import pandas as pd


    # Generate all combinations
    tuple_final = get_combinations_proposed(
        initial_df,
        num_participants,
        max_percentage,
        num_combinations,
        vars_to_eval,
        rules
    )


    df_kpis = pd.DataFrame( columns=['Metric', 
                                    'AVG Aproval Rate', 
                                    'BIC Approval Rate',
                                    'AVG BPS', 
                                    'BIC BPS',
                                    ])

    for var in vars_to_eval:
        try:
            selected_comb = int(selected_comb)
        except ValueError:
            print("Error: Please enter a valid integer.")

        result_break = get_comb_kpis(var, tuple_final ,selected_comb, bic_apr_q = 0.85 )
        if not result_break:
            continue
        series_to_append = pd.Series(result_break, index=df_kpis.columns)
        df_kpis = pd.concat([df_kpis, series_to_append.to_frame().T], ignore_index=True)
    
    return df_kpis

def save_dfs_to_excel(list_of_dfs, sheet_names, file_name="output.xlsx"):
    """
    Saves a list of pandas DataFrames to a single Excel file, with each
    DataFrame on a separate sheet.

    Args:
        list_of_dfs (list): A list of pandas DataFrame objects.
        sheet_names (list): A list of strings for the respective sheet names.
        file_name (str): The name of the output Excel file.
    """
    if len(list_of_dfs) != len(sheet_names):
        raise ValueError("The number of DataFrames must match the number of sheet names.")

    # Use the ExcelWriter context manager to ensure the file is properly saved and closed.
    try:
        with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
            for i, df in enumerate(list_of_dfs):
                # Write each DataFrame to a specific sheet.
                # The index=False argument prevents pandas from writing the
                # DataFrame index as a column in the Excel sheet.
                sheet_name = sheet_names[i]
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"Successfully saved data to '{os.path.abspath(file_name)}'")
    except Exception as e:
        print(f"An error occurred: {e}")

# Datasets

In [18]:
# Geral
df_geral = df_raw

# Selected Industry Flag
df_selected_flag = df_raw.copy()
industries = [
    "Restaurants",
    "Accommodations",
    "Groceries",
    "General Retail",
    "Fuel & Auto",
    "Finance and Insurance ",
    "Transport & Warehousing",
    "Consumer Electronics",
    "Entertainment & Recreation",
    "Apparel",
    "Other Services",
    "Public Administration",
    "Telco & Network ",
    "Travel Agencies & Remediation Services",
    "Wholesale Trade",
    "Professional Services",
    "Sports & Hobby",
    "Miscellaneous",
    "Education Services",
    "Construction Services"
]

selected_industries = ["Others", "Miscellaneous", "Consumer Electronics", "Travel Agencies & Remediation Services", "Accommodations", "Finance and Insurance ", "Telco & Network "]
df_selected_flag.loc[~df_selected_flag["super_industry_name_grouped"].isin(industries), "super_industry_name_grouped"] = "Others"
df_selected_flag.loc[~df_selected_flag["super_industry_name_grouped"].isin(selected_industries), "super_industry_name_grouped"] = "DISCARD"
df_selected_flag.loc[df_selected_flag["super_industry_name_grouped"].isin(["Travel Agencies & Remediation Services", "Accommodations"]), "super_industry_name_grouped"] = "T&E"
df_selected_flag["super_industry_name_grouped"].value_counts(dropna=False)
df_selected_flag['selected_industry'] = df_selected_flag.super_industry_name_grouped.apply(lambda x: 'SELECTED' if x != 'DISCARD' else 'REST')

# Selected Industries Filtered
df_selected = df_selected_flag.copy()
df_selected = (
    df_selected.loc[
        df_selected.super_industry_name_grouped != 'DISCARD'
    ]  
)

In [ ]:
target_breaks = {
    #Geral
    'Mensal': (['month_year'], df_geral, 5),
    'Ticket Range': (['ticket_range'], df_geral, 5),
    'Period': (['periodo_txn'], df_geral, 5),
    
    'Selected Industry Geral': (['selected_industry'], df_selected_flag, 5),
    'Industry Geral': (["super_industry_name_grouped"], df_selected, 5),
    
    'Domestic': (['flag_domestic'], df_geral, 1),
    'Selected Industry x Domestic': (["flag_domestic","selected_industry"], df_selected_flag, 1),
    'Industry x Domestic': (["flag_domestic","super_industry_name_grouped"], df_selected, 1),
    
    'Modo de Entrada': (['pan_entry_mode'], df_geral, 1),
    'Selected Industry x Modo de Entrada': (["selected_industry","pan_entry_mode"], df_selected_flag, 1),
    'Industry x Modo de Entrada': (["super_industry_name_grouped","pan_entry_mode"], df_selected, 1),

    # Ext 2
    #'Click to Pay Geral': (['click_to_pay'], df_geral, 5),
    'DI Geral': (['range_score_di'], df_geral, 1),
    'DTI Geral': (['dti_score'], df_geral, 1),

    # Geral
    'Token Geral': (['token_ind'], df_geral, 1),
    'Wallet Geral': (['wallet_flag'], df_geral, 5),

    #'Selected Industry x Click to Pay': (['click_to_pay','selected_industry'], df_selected_flag, 5),
    'Selected Industry x Token': (['token_ind','selected_industry'], df_selected_flag, 1),
    'Selected Industry x Wallet': (['wallet_flag','selected_industry'], df_selected_flag, 5),

    #'Industry x Click to Pay': (['click_to_pay','super_industry_name_grouped'], df_selected, 5),
    'Industry x Token': (['token_ind','super_industry_name_grouped'], df_selected, 1),
    'Industry x Wallet': (['wallet_flag','super_industry_name_grouped'], df_selected, 5),

    #'Mensal x Click to Pay': (['click_to_pay','month_year'], df_geral, 5),
    'Mensal x Token': (['token_ind','month_year'], df_geral, 1),
    'Mensal x Wallet': (['wallet_flag','month_year'], df_geral, 5),
}



# Processing

In [20]:
peers_dict = {
    1: [
        'BANCO COOPERATIVO SICREDI SA',
        'BANCO C6 SA',
        'BANCO BTG PACTUAL SA',
        'BANCO BRADESCO S.A.'
    ],
    2: [
        'BANCO INTER S.A.',
        'BANCO C6 SA',
        'BANCO BTG PACTUAL SA',
        'BANCO BRADESCO S.A.'
    ],
    3: [
        'ITAU UNIBANCO S.A.',
        'BANCO C6 SA',
        'BANCO BTG PACTUAL SA',
        'BANCO BRADESCO S.A.'
    ],
    4: [
        'BANCO INTER S.A.',
        'BANCO COOPERATIVO SICREDI SA',
        'BANCO BTG PACTUAL SA',
        'BANCO BRADESCO S.A.'
    ],
    5: [
        'ITAU UNIBANCO S.A.',
        'BANCO COOPERATIVO SICREDI SA',
        'BANCO BTG PACTUAL SA',
        'BANCO BRADESCO S.A.'
    ]
}

In [21]:
# Configuration
rows = ['issuer_name']
columns_pivot = ["app", "txn", "fraud"]
dict2convert = {"Approved": "app", "Total": "txn", "Fraud": "fraud"}
issuer_column = "issuer_name"
issuer_name = "BANCO SANTANDER (BRASIL) S.A."

# Benchmark rules and parameters
rules = [(5, 26), (6, 31), (7, 36), (10, 41)]
num_participants = 4
max_percentage = 35
num_combinations = 5

kpi_results = {
    name: (
        balance_peer(
        params[1],
        rows,
        params[0],
        columns_pivot,
        dict2convert,
        issuer_column,
        issuer_name,
        rules,
        num_combinations,
        num_participants,
        max_percentage,
        params[2]
        ),
        f'{params[2]}:\n{peers_dict[params[2]]}'
    )
    for name, params in target_breaks.items()
}



Brakes to process: ['month_year']
iss_name     : BANCO SANTANDER (BRASIL) S.A.
iss_var_name : issuer_name
breaks       : ['General', '2025-01', '2025-02', '2025-03', '2025-04']
cnt_amt_flag : Cnt
None
 >> DF input values in df_input
 >> Breaks calculated in df_breaks
 >> Distance with peers in peer_distances
(6, 16) 
 (0, 0) 
 (6, 21) 
 (5, 22) 
 ['General', '2025-01', '2025-02', '2025-03', '2025-04']
[(1, 'v_General_Cnt_br0'), (5, 'v_2025-01_Cnt_br1'), (9, 'v_2025-02_Cnt_br2'), (13, 'v_2025-03_Cnt_br3'), (17, 'v_2025-04_Cnt_br4')]
initial_df
EVALUATING v_General_Cnt_br0:
Unable to find more combinations.
No combinations under rule choose. Weighting: 
1
3
3
3
4

Validated Data:
Combination 1:
            Issuer Name  Distance to Client  v_General_Cnt_br0  \
3      BANCO INTER S.A.                   0         1080487.06   
2           BANCO C6 SA                   0         1080487.06   
0   BANCO BRADESCO S.A.                   0          938768.00   
1  BANCO BTG PACTUAL SA         

c:\Users\e176097\OneDrive - Mastercard\Documents\Santander\Peer Benchmark Tool\AAnF_Benchmark_Tool.py:988: RuntimeWarning: divide by zero encountered in scalar divide
  avg_apr_rate  = summary_df[ap_name].sum() / summary_df[tl_name].sum() #.sum()/summary_df[tl_name].sum()
C:\Users\e176097\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_function_base_impl.py:4653: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
c:\Users\e176097\OneDrive - Mastercard\Documents\Santander\Peer Benchmark Tool\AAnF_Benchmark_Tool.py:988: RuntimeWarning: divide by zero encountered in scalar divide
  avg_apr_rate  = summary_df[ap_name].sum() / summary_df[tl_name].sum() #.sum()/summary_df[tl_name].sum()
C:\Users\e176097\AppData\Roaming\Python\Python312\site-packages\numpy\lib\_function_base_impl.py:4653: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)




Brakes to process: ['token_ind']
iss_name     : BANCO SANTANDER (BRASIL) S.A.
iss_var_name : issuer_name
breaks       : ['General', 'Non-tokenized', 'Tokenized']
cnt_amt_flag : Cnt
None
 >> DF input values in df_input
 >> Breaks calculated in df_breaks
 >> Distance with peers in peer_distances
(6, 10) 
 (0, 0) 
 (6, 13) 
 (5, 14) 
 ['General', 'Non-tokenized', 'Tokenized']
[(1, 'v_General_Cnt_br0'), (5, 'v_Non-tokenized_Cnt_br1'), (9, 'v_Tokenized_Cnt_br2')]
initial_df
EVALUATING v_General_Cnt_br0:
Unable to find more combinations.
No combinations under rule choose. Weighting: 
1
3
3
3
4

Validated Data:
Combination 1:
            Issuer Name  Distance to Client  v_General_Cnt_br0  \
3      BANCO INTER S.A.                   0         1080487.06   
2           BANCO C6 SA                   0         1080487.06   
0   BANCO BRADESCO S.A.                   0          938768.00   
1  BANCO BTG PACTUAL SA                   0           78161.00   

   v_General_Cnt_br0_Percentage  Factor_

# Final Results

## Inspection

In [22]:
for name, (df, comb) in kpi_results.items():
    print(name)
    print('Combination', comb)
    display(df)
    print('########################################\n\n')

Mensal
Combination 5:
['ITAU UNIBANCO S.A.', 'BANCO COOPERATIVO SICREDI SA', 'BANCO BTG PACTUAL SA', 'BANCO BRADESCO S.A.']


,Metric,AVG Aproval Rate,BIC Approval Rate,AVG BPS,BIC BPS
0,v_General_Cnt_br0,0.58,0.82,30.87,15.90
1,v_2025-01_Cnt_br1,0.58,0.81,35.54,29.51
2,v_2025-02_Cnt_br2,0.60,0.84,34.48,17.68
3,v_2025-03_Cnt_br3,0.57,0.82,30.93,12.02
4,v_2025-04_Cnt_br4,0.57,0.82,23.34,7.57


########################################


Ticket Range
Combination 5:
['ITAU UNIBANCO S.A.', 'BANCO COOPERATIVO SICREDI SA', 'BANCO BTG PACTUAL SA', 'BANCO BRADESCO S.A.']


,Metric,AVG Aproval Rate,BIC Approval Rate,AVG BPS,BIC BPS
0,v_General_Cnt_br0,0.58,0.82,30.87,15.90
1,v_01.0-15_Cnt_br1,0.57,0.80,25.83,7.02
2,v_02.15-50_Cnt_br2,0.55,0.85,27.92,16.20
3,v_03.50-100_Cnt_br3,0.59,0.85,32.86,24.37
4,v_04.100-200_Cnt_br4,0.60,0.82,37.09,19.36
5,v_05.200-500_Cnt_br5,0.66,0.83,39.27,22.17
6,v_06.500-1000_Cnt_br6,0.66,0.82,36.26,14.67
7,v_07.1000-2000_Cnt_br7,0.64,0.81,30.10,15.32
8,v_08. > 2000_Cnt_br8,0.55,0.75,32.06,7.96


########################################


Period
Combination 5:
['ITAU UNIBANCO S.A.', 'BANCO COOPERATIVO SICREDI SA', 'BANCO BTG PACTUAL SA', 'BANCO BRADESCO S.A.']


...

########################################


Mensal x Token
Combination 1:
['BANCO COOPERATIVO SICREDI SA', 'BANCO C6 SA', 'BANCO BTG PACTUAL SA', 'BANCO BRADESCO S.A.']


,Metric,AVG Aproval Rate,BIC Approval Rate,AVG BPS,BIC BPS
0,v_General_Cnt_br0,0.57,0.82,22.52,11.00
1,v_Non-tokenized_2025-01_Cnt_br1,0.46,0.77,34.59,24.30
2,v_Non-tokenized_2025-02_Cnt_br2,0.48,0.80,29.52,9.13
3,v_Non-tokenized_2025-03_Cnt_br3,0.45,0.76,28.25,7.20
4,v_Non-tokenized_2025-04_Cnt_br4,0.46,0.79,23.26,5.49
5,v_Tokenized_2025-01_Cnt_br5,0.75,0.92,24.48,21.68
6,v_Tokenized_2025-02_Cnt_br6,0.76,0.94,23.46,15.03
7,v_Tokenized_2025-03_Cnt_br7,0.74,0.93,23.07,9.00
8,v_Tokenized_2025-04_Cnt_br8,0.73,0.92,16.59,5.12


########################################


Mensal x Wallet
Combination 5:
['ITAU UNIBANCO S.A.', 'BANCO COOPERATIVO SICREDI SA', 'BANCO BTG PACTUAL SA', 'BANCO BRADESCO S.A.']


,Metric,AVG Aproval Rate,BIC Approval Rate,AVG BPS,BIC BPS
0,v_General_Cnt_br0,0.58,0.82,30.87,15.90
1,v_NON-WALLET_2025-01_Cnt_br1,0.48,0.76,43.92,35.59
2,v_NON-WALLET_2025-02_Cnt_br2,0.49,0.78,43.56,21.06
3,v_NON-WALLET_2025-03_Cnt_br3,0.46,0.75,37.29,15.55
4,v_NON-WALLET_2025-04_Cnt_br4,0.47,0.78,29.45,9.97
5,v_WALLET_2025-01_Cnt_br5,0.58,0.81,35.54,29.51
6,v_WALLET_2025-02_Cnt_br6,0.60,0.84,34.48,17.68
7,v_WALLET_2025-03_Cnt_br7,0.57,0.82,30.93,12.02
8,v_WALLET_2025-04_Cnt_br8,0.57,0.82,23.34,7.57


########################################




## Saving to excel

In [23]:
save_dfs_to_excel(
    list_of_dfs=[df[0] for df in kpi_results.values()],
    sheet_names=[f'C{comb[0]}|{name}' for name, (_, comb) in kpi_results.items()],
    file_name='20250625_santander_peer_group_data_pj_peer_v2.xlsx'
)

C:\Users\e176097\AppData\Roaming\Python\Python312\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Successfully saved data to 'c:\Users\e176097\OneDrive - Mastercard\Documents\Santander\Peer Benchmark Tool\20250625_santander_peer_group_data_pj_peer_v2.xlsx'
